# 00 — EDA

10 collections, 150 to 3M rows. Checks watermark `created_at`, nulls, orphans, gaps. Fixture-only fallback.

In [1]:
import pathlib
import json, pathlib, glob, pandas as pd, numpy as np
# load fixtures
samples = {}
for f in sorted(glob.glob("tests/data/*.json") if pathlib.Path("tests/data").exists() else glob.glob("../tests/data/*.json")):
    name = pathlib.Path(f).stem
    data = json.loads(pathlib.Path(f).read_text()); samples[name] = data[0] if isinstance(data, list) and data else data
samples.keys()


dict_keys(['accounts', 'branches', 'card_transactions', 'cards', 'customers', 'employees', 'loan_payments', 'loans', 'support_tickets', 'transactions'])

In [2]:
# profiling table
df_prof = pd.DataFrame([
    ["customers", 60000, "customer_id", "created_at"],
    ["accounts", 95000, "account_id", "created_at"],
    ["transactions", 2000000, "transaction_id", "created_at+_id"],
    ["branches", 150, "branch_id", "none"],
    ["loans", 22000, "loan_id", "created_at"],
    ["cards", 65000, "card_id", "created_at"],
    ["card_transactions", 3000000, "card_txn_id", "created_at+_id"],
    ["loan_payments", 600000, "payment_id", "created_at"],
    ["support_tickets", 25000, "ticket_id", "created_at"],
    ["employees", 1800, "employee_id", "created_at"],
], columns=["collection","est_count","key","watermark"])
df_prof

,collection,est_count,key,watermark
0,customers,60000,customer_id,created_at
1,accounts,95000,account_id,created_at
2,transactions,2000000,transaction_id,created_at+_id
3,branches,150,branch_id,none
4,loans,22000,loan_id,created_at
5,cards,65000,card_id,created_at
6,card_transactions,3000000,card_txn_id,created_at+_id
7,loan_payments,600000,payment_id,created_at
8,support_tickets,25000,ticket_id,created_at
9,employees,1800,employee_id,created_at


In [3]:
# null check per sample
for name, doc in samples.items():
    nulls = [k for k,v in doc.items() if v is None]
    print(name, "nulls:", nulls if nulls else "none", "| has created_at:", "created_at" in doc)
# contracts expect no nulls on keys, 100% created_at


accounts nulls: none | has created_at: True
branches nulls: none | has created_at: True
card_transactions nulls: none | has created_at: True
cards nulls: none | has created_at: True
customers nulls: none | has created_at: True
employees nulls: none | has created_at: True
loan_payments nulls: none | has created_at: True
loans nulls: none | has created_at: True
support_tickets nulls: none | has created_at: True
transactions nulls: none | has created_at: True


In [4]:
# duplicate keys check (business key unique per docs/profiling.md)
for name, doc in samples.items():
# single-doc fixture -> check dups via count vs distinct
    print(name, "key unique in sample: single doc, expect 0 dups in prod")
# live: db.coll.aggregate group+match for dups


accounts key unique in sample: single doc, expect 0 dups in prod
branches key unique in sample: single doc, expect 0 dups in prod
card_transactions key unique in sample: single doc, expect 0 dups in prod
cards key unique in sample: single doc, expect 0 dups in prod
customers key unique in sample: single doc, expect 0 dups in prod
employees key unique in sample: single doc, expect 0 dups in prod
loan_payments key unique in sample: single doc, expect 0 dups in prod
loans key unique in sample: single doc, expect 0 dups in prod
support_tickets key unique in sample: single doc, expect 0 dups in prod
transactions key unique in sample: single doc, expect 0 dups in prod


In [5]:
# orphan rows: FK check on samples
# accounts.customer_id -> customers, transactions.account_id -> accounts
acc = samples.get("accounts", {})
cust = samples.get("customers", {})
txn = samples.get("transactions", {})
print("accounts.customer_id", acc.get("customer_id"), "exists in customers?", acc.get("customer_id")==cust.get("customer_id"))
print("transactions.account_id", txn.get("account_id"), "exists in accounts?", txn.get("account_id")==acc.get("account_id"))
# orphan check: LEFT JOIN where FK IS NULL = 0


accounts.customer_id 11749 exists in customers? False
transactions.account_id 42369 exists in accounts? False


In [6]:
# numeric distributions - use fixtures + plotly boxplot
import plotly.express as px
# collect numeric fields from samples
rows = []
for name in ["customers","accounts","transactions","cards","card_transactions","loans","loan_payments"]:
    doc = samples.get(name, {})
    for k in ["annual_income","credit_score","balance","amount","credit_limit","loan_amount","amount_paid"]:
        if k in doc and isinstance(doc[k], (int,float)):
            rows.append({"collection": name, "field": k, "value": doc[k]})
df_num = pd.DataFrame(rows)
df_num
# boxplot (tiny sample -> shape only, real run on Mongo/Spark)
if not df_num.empty:
    fig = px.box(df_num, x="field", y="value", color="collection", points="all")
    fig.update_layout(height=350, title="Value spread by field (sample)")
    fig.show()
else:
    print("no numeric in samples - run on live lake for boxplot")


In [7]:
# amount / balance spread - live lake if up
try:
    from jobs.common.spark import get_spark
    spark = get_spark("eda")
    # try silver/bronze counts if lake exists
    for tbl in ["banking.silver.transactions","banking.silver.card_transactions","banking.silver.customers"]:
        try:
            cnt = spark.table(tbl).count()
            print(tbl, cnt)
            # boxplot via pandas
            pdf = spark.table(tbl).select("amount").dropna().limit(5000).toPandas() if "amount" in [f.name for f in spark.table(tbl).schema.fields] else None
            if pdf is not None and not pdf.empty:
                import plotly.express as px
                fig = px.box(pdf, y="amount", points="outliers")
                fig.update_layout(title=f"{tbl} amount boxplot")
                fig.show()
        except Exception as e:
            print(tbl, "skip", str(e)[:100])
    spark.stop()
except Exception as e:
    print("lake not up - sample boxplot above is shape only:", str(e)[:120])
# gaps: amount <=0, negative, null rate drift


lake not up - sample boxplot above is shape only: No module named 'jobs'


In [8]:
# bar: collection volume (est)
import plotly.express as px
fig = px.bar(df_prof.sort_values("est_count"), x="est_count", y="collection", orientation="h", log_x=True)
fig.update_layout(title="Est. volume by collection (log scale)", height=400)
fig.show()
# gaps: 150 branches vs 3M card_transactions - partitioning differs


In [9]:
# fault injection preview
import glob
for f in sorted(glob.glob("tests/data/fault_injection/*")):
    print(pathlib.Path(f).name, "-", pathlib.Path(f).read_text()[:100])
# these should be quarantined, not reach Gold
